# Análise Exploratória de Dados: Aplicativos da Google Play Store

### Introdução

Este notebook realiza uma análise exploratória sobre um conjunto de dados de aplicativos da Google Play Store. O objetivo é extrair insights sobre as características dos aplicativos, como as categorias mais populares, a relação entre avaliações e instalações, e outros fatores que podem influenciar o sucesso de um aplicativo na plataforma.

### O Conjunto de Dados

O dataset utilizado, `googleplaystore_cleaned.csv`, é uma versão tratada contendo informações públicas de milhares de aplicativos. As principais colunas incluem:
* **App**: Nome do aplicativo.
* **Category**: Categoria à qual o aplicativo pertence.
* **Rating**: A avaliação média do aplicativo.
* **Reviews**: O número total de avaliações.
* **Size**: O tamanho do aplicativo em megabytes (MB).
* **Installs**: O número aproximado de instalações.
* **Type**: Se o aplicativo é Gratuito (`Free`) ou Pago (`Paid`).
* **Price**: O preço do aplicativo.
* **Content Rating**: Classificação indicativa do conteúdo.
* **Genres**: Gêneros do aplicativo.
* **Last Updated**: Data da última atualização do aplicativo.
* **Current Ver**: Versão atual do aplicativo.
* **Android Ver**: Versão mínima do Android necessária para instalar o aplicativo.

## 1. Configuração do Ambiente e Carregamento dos Dados

A célula de código a seguir é responsável por preparar nosso ambiente de trabalho. As seguintes ações são executadas:

1.  **Importação de Bibliotecas**: Carregamos as ferramentas essenciais para a análise:
    * `pandas`: Para a manipulação e análise dos dados em formato de tabela (DataFrame).
    * `matplotlib`: Para a criação de gráficos e visualizações.
    * `os`, `sys`, `pathlib`: Para gerenciar caminhos de arquivos de forma robusta, tornando o notebook mais portável.

2.  **Configuração do Caminho (Path)**: O bloco de código que manipula o `sys.path` é um passo importante para a organização do projeto. Ele permite que o notebook encontre o arquivo CSV que está em um diretório-pai chamado `dados`, sem a necessidade de usar caminhos fixos (hardcoded), que poderiam quebrar em outros computadores.

3.  **Leitura dos Dados**: Por fim, a função `pd.read_csv` lê o arquivo e carrega seu conteúdo em um DataFrame do pandas chamado `ds`. Esta variável `ds` será a nossa principal fonte de dados para todas as análises subsequentes.

In [ ]:
import os
import sys

from matplotlib import pyplot as plt, ticker
import pandas as pd

# Notebook não encontrava dataset, então adicionei o caminho do módulo ao sys.path
# Isso é necessário para que o Python possa localizar o módulo 'dados' corretamente.
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from pathlib import Path
DATA_DIR = Path(module_path) / 'dados'

ds = pd.read_csv(DATA_DIR / 'googleplaystore_cleaned.csv')

## 2. Análise do Top 5 Aplicativos Mais Instalados

Nesta seção, o objetivo é identificar os 5 aplicativos com o maior número de instalações. Uma análise inicial revelou que vários aplicativos atingem a marca máxima de "1 bilhão de instalações", tornando necessário um critério de desempate. Foi adotado o número de `Reviews` para ordenar esses aplicativos.

Para visualizar essa relação complexa, onde uma variável define o grupo (Instalações) e outra define a ordem dentro dele (Reviews), foi criado um **gráfico de barras agrupado com eixo duplo**.

### Metodologia do Gráfico

O gráfico é construído para comparar duas métricas com escalas muito diferentes em um mesmo visual:
- **Eixo Y Esquerdo (azul)**: Representa o número de **Instalações**, com escala em bilhões.
- **Eixo Y Direito (verde)**: Representa o número de **Reviews**, com escala em milhões.

O código executa os seguintes passos:
1.  **Ordenação e Seleção**: O DataFrame é ordenado primeiro por `Installs` e depois por `Reviews` para obter o ranking correto dos 5 melhores aplicativos.
2.  **Criação de Eixo Duplo**: A função `ax1.twinx()` cria um segundo eixo Y (`ax2`) que compartilha o mesmo eixo X, permitindo a plotagem de duas escalas diferentes.
3.  **Posicionamento das Barras**: As barras são posicionadas lado a lado (`x - width/2` e `x + width/2`) para cada aplicativo, facilitando a comparação direta.
4.  **Customização dos Eixos**: Cada eixo é formatado individualmente com cores e rótulos correspondentes às suas barras. O `FuncFormatter` é usado para abreviar os números grandes (ex: `1B` para 1 bilhão, `27M` para 27 milhões), melhorando a legibilidade.
5.  **Legenda Unificada**: Uma legenda clara é gerada para indicar qual cor corresponde a qual métrica.

O resultado é um gráfico que não apenas mostra *quais* são os aplicativos mais instalados, mas explica visualmente o *porquê* de sua ordenação, destacando a importância dos reviews como fator de desempate.

In [ ]:
# Gera um gráfico de barras horizontal para os 5 apps com maior número de instalações
import numpy as np

ds_sorted = ds.sort_values(by=['Installs', 'Reviews'], ascending=[False, False])
top_5_apps = ds_sorted.head(5)

app_names = top_5_apps['App']
x = np.arange(len(app_names))
width = 0.4

fig, ax1 = plt.subplots(figsize=(14, 8))
ax2 = ax1.twinx()

color1 = 'cornflowerblue'
rects1 = ax1.bar(x - width/2, top_5_apps['Installs'], width, label='Instalações', color=color1)

color2 = 'mediumseagreen'
rects2 = ax2.bar(x + width/2, top_5_apps['Reviews'], width, label='Reviews', color=color2)

ax1.set_ylabel('Instalações (em Bilhões)', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda val, pos: f'{int(val/1_000_000_000)}B'))
ax1.set_ylim(0, top_5_apps['Installs'].max() * 1.1)

ax2.set_ylabel('Reviews (em Milhões)', color=color2, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda val, pos: f'{val/1_000_000:.0f}M'))
ax2.set_ylim(0, top_5_apps['Reviews'].max() * 1.1)

ax1.set_title('Top 5 Apps Mais Instalados (Desempate por Nº Review)', fontsize=16, pad=20)
ax1.set_xticks(x)
ax1.set_xticklabels(app_names, rotation=45, ha='right')

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper center')

fig.tight_layout()
plt.show()

## 3. Distribuição de Aplicativos por Categoria

Nesta etapa, foi analisado a distribuição dos aplicativos entre as diversas categorias disponíveis na Play Store. O objetivo é entender quais são as categorias com maior representatividade no mercado, ou seja, com o maior número de aplicativos publicados.

Para isso, utilizamos um gráfico de pizza, que é ideal para mostrar a proporção de cada parte em relação a um todo.

### Metodologia e Tratamento dos Dados

Um desafio comum em gráficos de pizza é o excesso de "fatias" pequenas, que poluem o visual e dificultam a interpretação. Para solucionar isso, adotamos uma abordagem estratégica antes de gerar o gráfico:

1.  **Contagem Inicial**: Primeiro, conta o número de aplicativos em cada categoria usando `value_counts()`.
2.  **Agrupamento Estratégico**:
    * Seleciona as **9 categorias mais frequentes** para serem exibidas individualmente.
    * Soma a contagem de todas as categorias restantes e as agrupamos em uma única fatia consolidada, denominada **"Outras"**.
3.  **Geração do Gráfico**: Com os dados preparados em 10 fatias (as 9 principais + "Outras"), usa a função `plt.pie` para criar a visualização.
    * O parâmetro `autopct='%1.1f%%'` formata os valores para exibir o percentual de cada fatia com uma casa decimal.
    * `plt.axis('equal')` garante que o gráfico tenha um formato perfeitamente circular.

Essa metodologia resulta em um gráfico limpo e de fácil interpretação, que destaca as categorias dominantes sem perder a noção do peso combinado das categorias de nicho.

In [ ]:
# Gráfico de pizza para a distribuição de aplicativos por categoria
contagem_categorias = ds['Category'].value_counts()

limite = 9
principais_categorias = contagem_categorias.head(limite)
outras_categorias = pd.Series([contagem_categorias.iloc[limite:].sum()], index=['Outras'])

dados_para_grafico = pd.concat([principais_categorias, outras_categorias])

plt.figure(figsize=(12, 12))

plt.pie(
    dados_para_grafico,
    labels=dados_para_grafico.index.to_list(),
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.85
)

plt.title('Distribuição de Aplicativos por Categoria', fontsize=16)
plt.axis('equal')
plt.show()

## 4. Identificação do Aplicativo Mais Caro

Nesta análise, foi realizado uma busca simples para identificar qual é o aplicativo com o preço mais elevado listado no conjunto de dados. Este é um bom exercício para entender os extremos de valores (`outliers`) no dataset.

### Metodologia

O processo é realizado em três passos principais e eficientes:

1.  **Localização do Valor Máximo**: Em vez de apenas encontrar qual é o maior preço, podemos usar o método `.idxmax()`. Esta função do pandas é muito otimizada e retorna diretamente o **índice** da linha que contém o maior valor na coluna `Price`.

2.  **Extração dos Dados**: Com o índice do aplicativo mais caro em mãos, utiliza-se `ds.loc[]` para acessar a linha completa do DataFrame. Isso permite extrair não apenas o preço, mas também todas as outras informações do aplicativo, como seu nome.

O resultado é então impresso em uma frase formatada, apresentando o achado de forma clara e direta.

In [ ]:
# Encontrar o aplicativo mais caro
indice_app_mais_caro = ds['Price'].idxmax()

app_mais_caro = ds.loc[indice_app_mais_caro]

print("--- Aplicativo Mais Caro Encontrado ---")
print(f"O aplicativo mais caro é '{app_mais_caro['App']}', custando ${app_mais_caro['Price']:.2f}.")

## 5. Contagem por Classificação de Conteúdo: "Mature 17+"

Nesta célula, filtramos o dataset para responder a uma pergunta específica: Quantos aplicativos são direcionados a um público adulto, com a classificação de conteúdo "Mature 17+"?

Este tipo de contagem é útil para entender a demografia de público para a qual os desenvolvedores mais criam aplicativos.

### Metodologia: Filtragem Booleana

A abordagem utilizada é um exemplo clássico e muito performático de **filtragem booleana** no pandas.

1.  **Criação da Condição (Máscara Booleana)**: A primeira linha, `ds['Content Rating'] == 'Mature 17+'`, não filtra o DataFrame diretamente. Em vez disso, ela cria uma `Series` do pandas contendo apenas valores `True` ou `False`. Cada linha do dataset recebe `True` se a classificação do app for "Mature 17+" e `False` caso contrário. Esta série de `True`/`False` é conhecida como "máscara".

2.  **Contagem por Soma**: A segunda linha, `condicao.sum()`, é um método muito eficiente para contar os resultados. Ao aplicar uma operação matemática como `.sum()` a uma série booleana, o pandas automaticamente trata os valores `True` como `1` e os `False` como `0`. Portanto, a soma total nos dá a contagem exata de quantos `True` existem na série, que é precisamente o número de aplicativos que satisfazem nossa condição.

Este método é, em geral, mais rápido e consome menos memória do que outras alternativas, como filtrar o DataFrame e depois contar o número de linhas com `len()`.

In [ ]:
# Contagem de aplicativos classificados como 'Mature 17+'
condicao = ds['Content Rating'] == 'Mature 17+'

quantidade = condicao.sum()

print("--- Contagem de Aplicativos 'Mature 17+' ---")
print(f"O número de aplicativos classificados como 'Mature 17+' é: {quantidade}")

## 6. Top 10 Aplicativos por Engajamento (Número de Reviews)

Além do número de instalações, a quantidade de avaliações (`Reviews`) é um dos indicadores mais fortes do engajamento e da popularidade de um aplicativo. Um alto número de reviews sugere uma base de usuários ativa e disposta a interagir.

Nesta célula, identificamos os 10 aplicativos que mais geraram reviews na plataforma.

### Metodologia

O processo para obter este ranking é uma sequência de manipulação de dados padrão em `pandas`:

1.  **Ordenação dos Dados**: A função `ds.sort_values(by='Reviews', ascending=False)` é o passo principal. Ela reordena todo o DataFrame, colocando os aplicativos com o maior número de reviews no topo.

2.  **Seleção do Top 10**: Após a ordenação, o método `.head(10)` é utilizado para extrair apenas as 10 primeiras linhas, que correspondem aos aplicativos mais avaliados.

3.  **Formatação da Saída**: Para uma apresentação limpa e focada, realizamos duas ações:
    * Seleciona-se apenas as colunas de interesse (`App` e `Reviews`).
    * O comando `reset_index(drop=True)` remove o índice original do DataFrame (que estaria desordenado) e o substitui por um novo, sequencial. Em seguida, `resultado.index = resultado.index + 1` ajusta este novo índice para que o ranking comece em 1, em vez do padrão 0, tornando a lista mais intuitiva para o leitor.

4.  **Exibição**: Finalmente, `print(resultado.to_string())` é usado para garantir que a tabela seja impressa no console de forma completa e bem alinhada, sem truncar os resultados.

In [ ]:
# Ordenar aplicativos por número de reviews e exibir os 10 mais populares
ds_ordenado = ds.sort_values(by='Reviews', ascending=False)

top_10_apps = ds_ordenado.head(10)

resultado = top_10_apps[['App', 'Reviews']].reset_index(drop=True)
resultado.index = resultado.index + 1 

print("--- Top 10 Aplicativos por Número de Reviews ---")
print(resultado.to_string())

## 7. Desempenho: Top 5 Categorias por Avaliação Média

Além de saber quais categorias têm mais aplicativos, é interessante descobrir quais categorias contêm os aplicativos de **maior qualidade percebida**. Uma boa forma de medir isso é através da avaliação média (`Rating`) dos aplicativos.

Nesta análise, calcula-se a nota média para cada categoria para identificar quais tipos de aplicativos são, em geral, mais bem avaliados pelos usuários.

### Metodologia: Agrupamento e Agregação

A técnica central utilizada aqui é a de **agrupamento e agregação**, um dos recursos mais poderosos da biblioteca `pandas`.

1.  **Agrupar por Categoria (`groupby`)**: Primeiro, a função `ds.groupby('Category')` reorganiza o DataFrame, criando "grupos" ou "baldes" para cada categoria existente.

2.  **Calcular a Média (`mean`)**: Em seguida, para cada um desses grupos, o método `.mean()` é aplicado à coluna `Rating`. Isso calcula a média aritmética de todas as avaliações de aplicativos dentro daquela categoria específica. O resultado é uma nova `Series` onde os índices são os nomes das categorias e os valores são suas respectivas notas médias.

3.  **Ordenar e Selecionar**: Finalmente, com a lista de notas médias em mãos, usamos `sort_values(ascending=False)` para ordená-la do maior para o menor e `.head(5)` para extrair o nosso ranking final das 5 melhores categorias.

O resultado nos mostra quais nichos de mercado se destacam pela alta satisfação do usuário, independentemente do número total de aplicativos que possuem.

In [ ]:
# Ordena as categorias por média de avaliação (Rating)
media_rating_por_categoria = ds.groupby('Category')['Rating'].mean()

top_categorias = media_rating_por_categoria.sort_values(ascending=False)

resultado = top_categorias.head(5)

print("--- Top 5 Categorias por Média de Avaliação ---")
print(resultado.to_string())

## 8. Análise de Correlação: Reviews vs. Rating

Nesta seção, investiga-se a existência de uma relação estatística entre o engajamento de um aplicativo (medido pelo número de `Reviews`) e a sua avaliação (`Rating`). A análise busca responder à pergunta: "Um maior número de reviews está associado a uma avaliação maior ou menor?"

Para isso, calcula-se o **coeficiente de correlação de Pearson**. Este é um valor entre -1 e 1 que mede a força e a direção de uma relação *linear* entre duas variáveis.

### Metodologia

O processo executado pelo código é o seguinte:

1.  **Cálculo da Matriz**: O método `.corr()` é aplicado a um subconjunto do DataFrame que contém apenas as colunas `Reviews` e `Rating`. Este método calcula a correlação entre todos os pares de colunas fornecidos, resultando em uma "matriz de correlação".

2.  **Extração do Valor**: Da matriz gerada, extrai-se o valor específico que representa a correlação entre as duas variáveis de interesse (`Reviews` e `Rating`) utilizando o método `.loc[]`.

3.  **Exibição do Resultado**: O coeficiente resultante é exibido com quatro casas decimais para maior precisão.

In [ ]:
# Calcula a correlação entre o número de reviews e a avaliação (Rating)
matriz_correlacao = ds[['Reviews', 'Rating']].corr()

correlacao = matriz_correlacao.loc['Reviews', 'Rating']

print("\n--- Correlação entre Número de Reviews e Avaliação (Rating) ---")
print(f"O coeficiente de correlação é: {correlacao:.4f}")

## 9. Visualização de Desempenho: Gráfico das Melhores Categorias

Para complementar a análise de desempenho do item 7, esta célula cria uma visualização para o ranking das 5 categorias com maior avaliação média. Um gráfico de barras é, em geral, mais eficaz que uma tabela para comunicar rankings e diferenças de magnitude de forma rápida e intuitiva.

### Metodologia da Visualização

A construção do gráfico envolve várias etapas de preparação e customização para garantir clareza e impacto na apresentação da informação.

1.  **Preparação dos Dados**: Inicialmente, executa-se o mesmo cálculo de `groupby` e `mean` da análise anterior para obter as 5 melhores categorias. A linha `media_rating.iloc[::-1]` é um passo crucial que inverte a ordem dos dados. Isso garante que, no gráfico horizontal, a categoria com a maior nota apareça no topo, como é esperado em um ranking.

2.  **Construção do Gráfico (`barh`)**: Utiliza-se a função `plt.barh` para criar um gráfico de barras na horizontal. Esta orientação é preferível à vertical quando os rótulos do eixo (neste caso, os nomes das categorias) são longos, pois melhora a legibilidade e evita a sobreposição de texto.

3.  **Aprimoramentos Visuais**: Para tornar o gráfico mais informativo, aplica-se uma série de customizações:
    * **Rótulos de Dados**: Um laço `for` percorre cada barra para adicionar, via `plt.text`, o valor numérico exato da avaliação média ao final dela. Isso confere precisão à visualização.
    * **Escala Focada (`xlim`)**: O limite do eixo X é ajustado para focar no intervalo de notas específico das categorias do topo (ex: de 4.0 a 4.6). Esta técnica amplifica as pequenas diferenças entre elas, tornando a comparação visual muito mais clara.
    * **Grade (`grid`)**: Adiciona-se uma grade de fundo no eixo X para facilitar a leitura e a comparação do comprimento das barras.

O resultado é uma visualização limpa, que não apenas apresenta o ranking, mas também destaca a magnitude da diferença na satisfação do usuário entre as categorias de melhor desempenho.

In [ ]:
# Gráfico de barras horizontal para as 5 categorias com maior média de avaliação
media_rating = ds.groupby('Category')['Rating'].mean().sort_values(ascending=False).head(5)

dados_grafico = media_rating.iloc[::-1]

plt.figure(figsize=(10, 6))
bars = plt.barh(dados_grafico.index, dados_grafico, color='skyblue')

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.2f}', va='center')

plt.xlabel('Média de Avaliação (Rating)', fontsize=12)
plt.ylabel('Categoria', fontsize=12)
plt.title('Top 5 Categorias por Média de Avaliação', fontsize=15)
plt.xlim(4.0, dados_grafico.max() + 0.1) 
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()

## 10. Visualização de Correlação: Gráfico de Dispersão

Após calcular o coeficiente de correlação numérico, é fundamental visualizar a relação entre `Reviews` e `Rating` para confirmar as conclusões e investigar possíveis padrões não-lineares. O gráfico de dispersão (*scatter plot*) é a ferramenta ideal para esta tarefa, onde cada ponto no gráfico representa um único aplicativo.

### Metodologia da Visualização

A criação de um gráfico de dispersão eficaz para este conjunto de dados exige a aplicação de técnicas específicas para lidar com a distribuição dos dados.

1.  **O Desafio da Escala e a Solução Logarítmica (`xscale('log')`)**: A distribuição do número de `Reviews` é extremamente assimétrica: a grande maioria dos aplicativos tem poucos reviews, enquanto um pequeno número de apps acumula milhões. Se plotado em uma escala linear padrão, quase todos os pontos ficariam "amontoados" perto do eixo Y, tornando a visualização inútil.
    * **Solução**: A função `plt.xscale('log')` transforma o eixo X para uma **escala logarítmica**. Isso comprime os valores altos e expande os baixos, distribuindo os pontos de forma mais uniforme pelo gráfico e permitindo a análise em toda a faixa de dados.

2.  **Visualização de Densidade (`alpha`)**: O parâmetro `alpha=0.2` torna os pontos semitransparentes. Em áreas onde muitos pontos se sobrepõem, a cor se torna mais intensa. Esta técnica ajuda a identificar regiões de alta densidade de aplicativos e a contornar o problema da sobreposição de pontos (*overplotting*).

3.  **Grade e Rótulos Claros**: Adiciona-se uma grade (`grid`) para facilitar a leitura dos valores nos eixos. É de suma importância que o rótulo do eixo X indique explicitamente o uso da escala logarítmica para evitar qualquer interpretação equivocada do gráfico.

In [ ]:
# Gráfico de dispersão entre o número de reviews e a avaliação (Rating)
plt.figure(figsize=(12, 7))

plt.scatter(ds['Reviews'], ds['Rating'], alpha=0.2, color='green')

plt.xscale('log')

plt.xlabel('Número de Reviews (em escala logarítmica)', fontsize=12)
plt.ylabel('Avaliação (Rating)', fontsize=12)
plt.title('Relação entre Avaliação e Número de Reviews', fontsize=15)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

plt.tight_layout()